# Jacobian conjecture counterexample by LLM

Anton Antonov  
[RakuForPrediction at WordPress](https://rakuforprediction.wordpress.com)   
July 2026

### Introduction

The [Jacobian conjecture](https://en.wikipedia.org/wiki/Jacobian_conjecture) was (very) recently disproved by a [counterexample found by Anthropic's Fable 5](https://x.com/__alpoge__/status/2079028340955197566). [MS1]. This notebook demonstrates that counterexample using some of the built-in Raku functions and dedicated packages that facilitate symbolic computations, like ["CortexJS"](https://raku.land/zef:antononcube/CortexJS), [AAp1], ["Proc::ZMQed"](https://raku.land/zef:antononcube/Proc::ZMQed), [AAp2], and ["WWW::WolframAlpha"](https://raku.land/zef:antononcube/WWW::WolframAlpha), [AAp3]. We implement and use a simple Raku `find-instance` to find three different points which the counterexample polynomial 3-dimensional (3D) transformation maps into the same 3D point.

In the comments of [this X-post](https://x.com/__alpoge__/status/2079028340955197566) (and New Scientist article, [MS1]) it is pointed out that the counterexample found using [Anthropic's Fable 5](https://www.anthropic.com/claude/fable) Large Language Model (LLM).

**Remark:** The points found in this notebook are different than those in the[ X-post](https://x.com/__alpoge__/status/2079028340955197566) .

---

## Theory

In this section we provide a theoretical formulation of the Jacobian conjecture. For more a mathematically elaborated formulation see the Wikipedia article ["Jacobian conjecture"](https://en.wikipedia.org/wiki/Jacobian_conjecture), [Wk1].

Suppose $u$ and $v$ are polynomials of the variables $x$ and $y$. Consider the 2-dimensional (2D) transformation (i.e., a vector valued function) $F(x,y)\text{:=}(u[x,y], v[x,y])$. The Jacobian conjectures says that if the Jacobian determinant:

$$J = \frac{\partial u}{\partial x} \frac{\partial v}{\partial y} - \frac{\partial u}{\partial y} \frac{\partial v}{\partial x}$$

is a non-zero constant, then there must exist another pair of polynomials that invert transformation with the polynomials in $u$ and $v$.

Below we show that the conjecture is not true using the 3D transformation polynomials proclaimed in [this X post](https://x.com/__alpoge__/status/2079028340955197566), [MS1].

**Remark:** This notebook is the Raku version of the Wolfram Language (WL) notebook ["Jacobian conjecture counterexample by LLM"](https://community.wolfram.com/groups/-/m/t/3766129), [AAn1].

**Remark:** The Wolfram Desktop or Wolfram Engine is installed then the WL symbolic computations can be done via Raku-chatbook's `bash` magic cell using `wolframscript`.

----

## Setup

Here are the packages used in this notebook:

In [1]:
use CortexJS;
use LaTeX::Grammar;
use Proc::ZMQed;
use WWW::WolframAlpha;
use Mathematica::Serializer;

use JavaScript::D3;

**Remark:** The package ["WWW::WolframAlpha"](https://raku.land/zef:antononcube/WWW::WolframAlpha) is available in any [Raku-chatbook session](https://rakuforprediction.wordpress.com/2024/05/18/chatbook-new-magic-cells/#wolframalpha), [AA1].

Here we define a sub to render matrices and vectors via LaTeX in Markdown magic cells of Raku-chatbook:

In [2]:
sub latex-matrix(@data) {
    ['$$', '\\begin{bmatrix}', @data.map({ $_».subst('$',:g).join(' & ') }).join(' \\\\' ~ "\n"), '\\end{bmatrix}', '$$'].join("\n")
}

&latex-matrix

---

## Polynomial mapping and Jacobian matrix determinant

In this section the Anthropic's Fable 5 found counterexample is exemplified using corresponding definitions and invocations.

Definition of the counterexample polynomial mapping as a list of LaTeX strings:

In [3]:
#%markdown
my @poly-map =
     '(1 + x y)^3 z + y^2 (1 + x y) (4 + 3 x y)', 
     'y + 3 x (1 + x y)^2 z + 3 x y^2 (4 + 3 x y)', 
     '2 x - 3 x^2 y - x^3 z';

latex-matrix(@poly-map)

$$
\begin{bmatrix}
(1 + x y)^3 z + y^2 (1 + x y) (4 + 3 x y) \\
y + 3 x (1 + x y)^2 z + 3 x y^2 (4 + 3 x y) \\
2 x - 3 x^2 y - x^3 z
\end{bmatrix}
$$

The Jacobian matrix:

In [4]:
#%markdown
my @jacobian-matrix=
cross(@poly-map, <x y z>)
.map({ 'D_' ~ $_.tail ~ '{' ~ $_.head ~ '}' })
.map({ $_
    ==> parse-latex()
    ==> evaluate()
    ==> to-latex() })
.rotor(3);

latex-matrix(@jacobian-matrix)

$$
\begin{bmatrix}
6xy^4+3yz(xy+1)^2+7y^3 & 12y^3x^2+3xz(xy+1)^2+21xy^2+8y & (xy+1)^3 \\
6zx^2y^2+18xy^3+3z(xy+1)^2+6xyz+12y^2 & 6yzx^3+27x^2y^2+6zx^2+24xy+1 & 3x(xy+1)^2 \\
-3zx^2-6xy+2 & -3x^2 & -x^3
\end{bmatrix}
$$

The Jacobian matrix -- via [`wolframscript`](https://www.wolfram.com/wolframscript/) and the Wolfram Function Repository (WFR) entry [JacobianMatrix](https://resources.wolframcloud.com/FunctionRepository/resources/JacobianMatrix)):

In [5]:
#% markdown
my $proc = run 'wolframscript', '-code', "ResourceFunction[\"JacobianMatrix\"][{"\{@poly-map.join(', ')\}"}, \{x, y, z\}]//Simplify//Flatten", :out;
my $res = $proc.out.slurp(:close);
latex-matrix($res.split(/<[{},]>/, :skip-empty)».trim.rotor(3))

$$
\begin{bmatrix}
y*(6*x*y^3 + 3*z + 6*x*y*z + y^2*(7 + 3*x^2*z)) & 12*x^2*y^3 + 3*x*z + 3*x*y^2*(7 + x^2*z) + y*(8 + 6*x^2*z) & (1 + x*y)^3 \\
3*(6*x*y^3 + z + 4*x*y*z + y^2*(4 + 3*x^2*z)) & 1 + 24*x*y + 6*x^3*y*z + 3*x^2*(9*y^2 + 2*z) & 3*x*(1 + x*y)^2 \\
2 - 6*x*y - 3*x^2*z & -3*x^2 & -x^3
\end{bmatrix}
$$

The determinant is a constant:

In [6]:
'\\det' ~ latex-matrix(@jacobian-matrix).lines[1 .. *-2].join("\n")
==> parse-latex()
==> evaluate()
==> simplify()

-2

We can directly find the determinant via WFR's [JacobianDeterminant](https://resources.wolframcloud.com/FunctionRepository/resources/JacobianDeterminant):

In [7]:
#% markdown
my $proc = run 'wolframscript', '-code', "ResourceFunction[\"JacobianDeterminant\"][{"\{@poly-map.join(', ')\}"}, \{x, y, z\}]", :out;
my $res = $proc.out.slurp(:close);

-2


----

## Finding the points mapped into the same point

Using FindInstance to find triplets of triplets, i.e., $p_i\text{:=}\left(x_i,y_i,z_i\right), i\in [1,2,3]$ ∧ $p_i\text{!=}p_j\land i\neq j$, for which the polynomial map gives the same result. 

Searching all three points is too slow. We use a two-stage computation: we find first a solution with two different points, then we find a third point that is different that previously found two.

Define a function that computes the polynomial mapping:

In [ ]:
use MONKEY-SEE-NO-EVAL;
my $poly-map-expr = '[' ~ @poly-map.map({ latex-interpret($_, actions => 'RakuAST').DEPARSE.subst('x', '$x', :g).subst('y', '$y', :g).subst('z', '$z', :g) }).join(', ') ~ ']';
my &poly-map = EVAL('sub ($x, $y, $z) {' ~ $poly-map-expr ~ '}');

sub { }

Verify definition of `&poly-map`:

In [11]:
&poly-map(1.1, 0.2, 1/2)

[1.135332 3.27098 0.8085]

Search grid:

In [12]:
my $b = 6;
#my @grid3d = (-$b, -$b + 1 ... $b) X (-$b, -$b + 1/2 ... $b) X (-$b, -$b + 1/100 ... $b);
my @zs = (([\+] -1/18 xx 2_000).reverse, [\+] 1/18 xx 2_000).flat.grep(-$b ≤ * ≤ $b);
my @grid3d = (-$b, -$b + 1 ... $b) X (-1, -1/2, 0, 1/2, 1) X @zs;
say @grid3d.elems;

14040


In [13]:
deduce-type(@grid3d):tally

Tuple([Tuple([Atom((Int)), Atom((Int)), Atom((Rat))]) => 8424, Tuple([Atom((Int)), Atom((Rat)), Atom((Rat))]) => 5616], 14040)

Pre-compute polynomial mapping values:

In [14]:
my %poly-map-values = @grid3d.race(:4degree).map({ $_.Str => &poly-map(|$_) });
%poly-map-values.elems

14040

In [15]:
#my @fp = -6, 0, 1/18;
my @fp = 0, -1, -71/18;
say @grid3d.grep({ norm($_ <<->> @fp) ≤ 0.0001 });
say nearest(@grid3d, @fp, 3);

((0 -1 -3.944444))
((0 -1 -3.944444) (0 -1 -4) (0 -1 -3.888889))


Search for two point mapped into one:

In [16]:
my $k = 0;
my $degree = 4;
my @found;

@grid3d.race(:$degree, batch => ceiling(@grid3d.elems / $degree)).map( -> @p {
  
  my @mp = |%poly-map-values{@p.Str};
  
  if $k %% 500 { say (:$k) }
  
  $k++;

  for @grid3d -> @candidate {

    my @mpc = |%poly-map-values{@candidate.Str};

    if (@mpc <<->> @mp)».abs.max ≤ 0.0001 && @p ne @candidate {
      @found.push( %(:@p, :@candidate, :@mp) );
      say @found.tail;
    }   
  }
});

@found.elems

k => 0
k => 0
k => 0
k => 0
k => 500
{candidate => (-4 0.5 0.5), mp => [0 0.5 0], p => (0 0.5 -1)}
k => 1000
{candidate => (3 -1 1.222222), mp => [0.222222 -2 0], p => (-3 0 0.222222)}
{candidate => (-2 1 2), mp => [0 1 0], p => (0 1 -4)}
{candidate => (6 0 0.055556), mp => [0.055556 1 0], p => (0 1 -3.944444)}
k => 1500
{candidate => (-2 -1 -1.111111), mp => [0 -1 -0.888889], p => (4 -1 0.888889)}
{candidate => (-1 0.5 -2.5), mp => [0 0.5 -6], p => (-3 0.5 0.5)}
{candidate => (3 -0.5 0.722222), mp => [-0.027778 0 0], p => (-3 0.5 0.722222)}
k => 2000
{candidate => (0 -1 -3.944444), mp => [0.055556 -1 0], p => (-6 0 0.055556)}
{candidate => (0 -0.5 -1), mp => [0 -0.5 0], p => (4 -0.5 0.5)}
k => 2500
{candidate => (3 0 0.222222), mp => [0.222222 2 0], p => (-3 1 1.222222)}
k => 3000
{candidate => (3 -0.5 0.5), mp => [0 -0.5 6], p => (1 -0.5 -2.5)}
k => 3500
{candidate => (4 -1 0.888889), mp => [0 -1 -0.888889], p => (-2 -1 -1.111111)}
k => 4000
k => 4500
k => 5000
{candidate => (-5 0.5 

30

In [19]:
.say for @found

{candidate => (-4 0.5 0.5), mp => [0 0.5 0], p => (0 0.5 -1)}
{candidate => (3 -1 1.222222), mp => [0.222222 -2 0], p => (-3 0 0.222222)}
{candidate => (-2 1 2), mp => [0 1 0], p => (0 1 -4)}
{candidate => (6 0 0.055556), mp => [0.055556 1 0], p => (0 1 -3.944444)}
{candidate => (-2 -1 -1.111111), mp => [0 -1 -0.888889], p => (4 -1 0.888889)}
{candidate => (-1 0.5 -2.5), mp => [0 0.5 -6], p => (-3 0.5 0.5)}
{candidate => (3 -0.5 0.722222), mp => [-0.027778 0 0], p => (-3 0.5 0.722222)}
{candidate => (0 -1 -3.944444), mp => [0.055556 -1 0], p => (-6 0 0.055556)}
{candidate => (0 -0.5 -1), mp => [0 -0.5 0], p => (4 -0.5 0.5)}
{candidate => (3 0 0.222222), mp => [0.222222 2 0], p => (-3 1 1.222222)}
{candidate => (3 -0.5 0.5), mp => [0 -0.5 6], p => (1 -0.5 -2.5)}
{candidate => (4 -1 0.888889), mp => [0 -1 -0.888889], p => (-2 -1 -1.111111)}
{candidate => (-5 0.5 0.388889), mp => [0 0.5 1.111111], p => (1 0.5 -0.611111)}
{candidate => (-1 -0.5 -0.611111), mp => [0 -0.5 -1.111111], p => (5

Here we find a third point:

Combine the solutions into one:

Check:

All points are different:

---

## Plot

In this section we give a visual representation of the counterexample using a 3D contour plot -- the found points are plotted in red. Here we derive a 3D contour plot of the polynomial transformation:

Combine the 3D contour plot with the points found in the previous section (plotted in red):

---

## References

### Articles, blog posts

[AA1] Anton Antonov, ["Chatbook New Magic Cells"](https://rakuforprediction.wordpress.com/2024/05/18/chatbook-new-magic-cells/), (2024), [RakuForPrediction at WordPress](https://rakuforprediction.wordpress.com). 

[MS1] Matthew Sparkes, ["AI's solution to 87-year-old riddle takes mathematicians by surprise"](https://www.newscientist.com/article/2580374-ais-solution-to-87-year-old-riddle-takes-mathematicians-by-surprise/), (2026), New Scientist.

[MW1] MathWorld entry, ["Jacobian conjecture"](https://mathworld.wolfram.com/JacobianConjecture.html).

[Wk1] Wikipedia entry, ["Jacobian conjecture"](https://en.wikipedia.org/wiki/Jacobian_conjecture).

### Notebooks

[AAn1] Anton Antonov, ["Jacobian conjecture counterexample by LLM"](https://community.wolfram.com/groups/-/m/t/3766129), (2026), [Wolfram Community](https://community.wolfram.com). 

### Packages 

[AAp1] Anton Antonov, ["CortexJS"](https://github.com/antononcube/Raku-CortexJS), (2026), [GitHub/antononcube](https://github.com/antononcube).

[AAp2] Anton Antonov, ["Proc::ZMQed"](https://github.com/antononcube/Raku-Proc-ZMQed), (2022), [GitHub/antononcube](https://github.com/antononcube).

[AAp3] Anton Antonov, ["WWW::WolframAlpha"](https://github.com/antononcube/Raku-WWW-WolframAlpha), (2024), [GitHub/antononcube](https://github.com/antononcube).